# 00 - Setup: Create Virtual Environment and Prerequisites

This notebook creates an isolated Python virtual environment and walks through all prerequisites for this demo.

It also bootstraps and registers a Jupyter kernel for this demo so it appears in VS Code kernel options in future sessions.

**Estimated time:** 5-10 minutes

---

## What you need

| Requirement | Notes |
|---|---|
| Python | `>= 3.10` |
| Azure CLI | `>= 2.55` |
| Bicep extension | `az bicep install` |
| Azure subscription | contributor access to create resources |

---

## Step 1 - Create Isolated Virtual Environment and Register Kernel

Each demo has its own virtual environment to keep dependencies isolated and reproducible.

In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

# Determine venv location (in demo root)
demo_root = Path("..").resolve()
venv_path = demo_root / ".venv"
is_windows = sys.platform == "win32"
venv_python = venv_path / ("Scripts" if is_windows else "bin") / ("python.exe" if is_windows else "python")
venv_pip = venv_path / ("Scripts" if is_windows else "bin") / ("pip.exe" if is_windows else "pip")

demo_name = demo_root.name
kernel_name = re.sub(r"[^a-z0-9_-]", "-", demo_name.lower())
kernel_display_name = f"Python ({demo_name})"

print(f"Demo root: {demo_root}")
print(f"Virtual environment path: {venv_path}")
print(f"Kernel name: {kernel_name}\n")

# Create venv if it doesn't exist
if not venv_path.exists():
    print("Creating isolated virtual environment...")
    result = subprocess.run([sys.executable, "-m", "venv", str(venv_path)], capture_output=True, text=True)
    if result.returncode == 0:
        print("Virtual environment created")
    else:
        print(f"ERROR creating venv: {result.stderr}")
        sys.exit(1)
else:
    print("Virtual environment already exists")

# Verify venv python executable exists
if not venv_python.exists():
    print("ERROR: Python executable not found")
    sys.exit(1)
print("Virtual environment ready")

# Ensure notebook kernel packages exist in the demo venv
subprocess.run([str(venv_python), "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([str(venv_python), "-m", "pip", "install", "ipykernel"], check=True)

# Register a persistent kernelspec so VS Code can discover it later
subprocess.run(
    [
        str(venv_python),
        "-m",
        "ipykernel",
        "install",
        "--user",
        "--name",
        kernel_name,
        "--display-name",
        kernel_display_name,
    ],
    check=True,
 )
print(f"Kernel registered as: {kernel_display_name}")

# Store venv paths for later cells
os.environ["VENV_PYTHON"] = str(venv_python)
os.environ["VENV_PIP"] = str(venv_pip)

## VPN prerequisites (before Step 2)

- Azure VPN Client uses multi-tenant app ID `c632b3df-fb67-4d84-bdcf-b95ad541b5c8`. Tenant admin consent is a one-time prerequisite per tenant.
- Azure VPN Client on Windows requires build `17763+`.
- The next cell checks if the app is already consented in your tenant.


In [ ]:
import subprocess
import sys
from pathlib import Path

demo_root = Path("..").resolve()
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

from app.notebook_helpers import resolve_az_cli

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found. Install Azure CLI and rerun this cell.")

tenant_id = subprocess.run([az_cmd, "account", "show", "--query", "tenantId", "-o", "tsv"], capture_output=True, text=True, check=True).stdout.strip()
check = subprocess.run([az_cmd, "ad", "sp", "show", "--id", "c632b3df-fb67-4d84-bdcf-b95ad541b5c8"], capture_output=True, text=True)
if check.returncode == 0:
    print("✓ Azure VPN Client already consented in tenant")
else:
    print("✗ Needs tenant admin consent — forward this command to your admin: ")
    print(f"az ad sp create --id c632b3df-fb67-4d84-bdcf-b95ad541b5c8 --tenant {tenant_id}")


## Step 2 - Authenticate to Azure (Run after switching to the demo kernel)

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

demo_root = Path("..").resolve()
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

from app.notebook_helpers import resolve_az_cli

az_cmd = resolve_az_cli()
if not az_cmd:
    print("ERROR: Azure CLI not found from this kernel.")
    print("Install Azure CLI or add it to PATH, then rerun this cell.")
    sys.exit(1)

os.environ["AZ_CLI_CMD"] = az_cmd
print(f"Using Azure CLI: {az_cmd}")

# Check existing login first to avoid unnecessary interactive prompt
account_check = subprocess.run(
    [az_cmd, "account", "show", "--query", "id", "-o", "tsv"],
    capture_output=True,
    text=True,
 )

if account_check.returncode == 0 and account_check.stdout.strip():
    print("Already authenticated with Azure CLI")
else:
    print("No active Azure login found. Starting device-code login flow...")
    print("Follow the instructions shown in the output.")
    login_result = subprocess.run([az_cmd, "login", "--use-device-code"] )
    if login_result.returncode == 0:
        print("Successfully logged in to Azure")
    else:
        print("ERROR: Azure login failed")
        sys.exit(1)

# Show available subscriptions
subprocess.run([az_cmd, "account", "list", "--output", "table"], check=False)

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

demo_root = Path("..").resolve()
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

from app.notebook_helpers import resolve_az_cli

env_cmd = os.getenv("AZ_CLI_CMD")
if env_cmd and Path(env_cmd).exists():
    az_cmd = env_cmd
else:
    az_cmd = resolve_az_cli() or "az"

# Optional: set a specific subscription if needed
SUBSCRIPTION_ID = None

if SUBSCRIPTION_ID:
    subprocess.run([az_cmd, "account", "set", "--subscription", SUBSCRIPTION_ID], check=False)

subprocess.run([az_cmd, "account", "show", "--output", "table"], check=False)

## Step 3 – Install Python dependencies in virtual environment

Update `requirements.txt` in your demo folder to match the packages your demo needs.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Recompute venv path so this cell works even after switching kernels
demo_root = Path("..").resolve()
is_windows = sys.platform == "win32"
venv_pip_path = demo_root / ".venv" / ("Scripts" if is_windows else "bin") / ("pip.exe" if is_windows else "pip")
venv_pip = os.getenv("VENV_PIP", str(venv_pip_path))

if not Path(venv_pip).exists():
    print(f"ERROR: venv pip not found at {venv_pip}")
    print("Run Step 1 first to create the virtual environment.")
    sys.exit(1)

print(f"Using venv pip: {venv_pip}")

# Upgrade pip in venv
print("Upgrading pip in virtual environment...")
pip_upgrade = subprocess.run([venv_pip, "install", "--upgrade", "pip"])
if pip_upgrade.returncode == 0:
    print("pip upgraded")
else:
    print("WARNING: pip upgrade returned a non-zero exit code")

# Install from requirements.txt with live output
print("\nInstalling dependencies from requirements.txt...")
requirements_file = Path("../requirements.txt")
if requirements_file.exists():
    install_result = subprocess.run([venv_pip, "install", "-r", str(requirements_file)])
    if install_result.returncode == 0:
        print("All packages installed successfully")
    else:
        print("ERROR: package installation failed")
        sys.exit(1)
else:
    print(f"ERROR: requirements.txt not found at {requirements_file}")
    sys.exit(1)

---

## Setup Complete

Virtual environment created, kernel registered, and dependencies ready.

**Next Steps:**
1. In the notebook toolbar, click the kernel name (top-right) and choose **Select Another Kernel...**
2. Open **Jupyter Kernels** and select **Python (search01-private-endpoints)**
3. If it does not appear, run **Developer: Reload Window**, then reopen this notebook and try again
4. Continue with `01_deploy_infra.ipynb`
